In [1]:
import json
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. CARGA DEL MODELO
# Reemplaza con la ruta de tu modelo Qwen3-4B-Instruct-2507 si lo tienes local o en tu HF
model_id = "Qwen/Qwen3-4B-Instruct-2507"

print("Cargando modelo...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Modelo cargado exitosamente.\n")

# --- VARIABLES DE ENTORNO (El ancla temporal) ---
FECHA_HOY = "2026-09-01"
DIA_HOY = "martes"

# ==========================================
# 2. BASELINE: PROMPTING DIRECTO (Deliverable 1)
# ==========================================
def correr_baseline(texto_usuario):
    prompt_directo = f'Agendar esta reunión y devolver JSON: "{texto_usuario}"'
    messages = [{"role": "user", "content": prompt_directo}]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
    respuesta = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    return respuesta

# ==========================================
# 3. INTERVENCIÓN: SYSTEM PROMPT + FEW SHOT + CONTEXTO
# ==========================================
def correr_intervencion(texto_usuario):
    # INYECCIÓN DE CONTEXTO: Aquí resolvemos la "temporal reasoning" failure
    system_prompt = f"""Eres un asistente de agendamiento de calendario.
Hoy es {DIA_HOY}, {FECHA_HOY}. Debes resolver las fechas relativas basándote en hoy.
Extrae la información a un JSON con los campos: titulo, fecha (YYYY-MM-DD), hora_inicio (HH:MM), duracion_minutos, participantes (lista), ubicacion, recordatorio_minutos, zona_horaria (America/Santiago por defecto).
Si falta un dato, usa null o []. Responde SOLO con el JSON validado."""

    # FEW-SHOT: Aquí resolvemos la "format dependence" failure
    ejemplo_user = "Reunión mañana a las 10"
    ejemplo_assistant = """{
  "titulo": "Reunión",
  "fecha": "2026-09-02",
  "hora_inicio": "10:00",
  "duracion_minutos": null,
  "participantes": [],
  "ubicacion": null,
  "recordatorio_minutos": null,
  "zona_horaria": "America/Santiago"
}"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": ejemplo_user},
        {"role": "assistant", "content": ejemplo_assistant},
        {"role": "user", "content": texto_usuario}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
    respuesta = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

    # POST-PROCESAMIENTO: Forzar extracción limpia del JSON
    json_match = re.search(r'\{.*\}', respuesta, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except:
            return {"error": "JSON malformado"}
    return {"error": "No emitió JSON", "raw": respuesta}

# ==========================================
# 4. DEMOSTRACIÓN PARA EL VIDEO
# ==========================================
print("=== DEMOSTRACIÓN DE MEJORA (Frase 38) ===")
frase_38 = "Oye, agéndame una junta con la Dani pasado mañana a las 4 en la sala B-11"
print(f"Input: {frase_38}\n")

print("-> Ejecutando BASELINE (Direct Prompting)...")
resultado_base = correr_baseline(frase_38)
print(f"Salida Baseline:\n{resultado_base}\n")

print("-> Ejecutando SOLUCIÓN (System Prompt + Fecha Inyectada)...")
resultado_solucion = correr_intervencion(frase_38)
print(f"Salida Solución:\n{json.dumps(resultado_solucion, indent=2, ensure_ascii=False)}\n")

print("=== LECTURA DE LÍMITES (Frase 40 - Failure Case) ===")
frase_40 = "q onda, quedamos el viernes a las 7 en lo de siempre?"
print(f"Input: {frase_40}\n")
resultado_limite = correr_intervencion(frase_40)
print(f"Salida Solución en caso límite:\n{json.dumps(resultado_limite, indent=2, ensure_ascii=False)}")
# En tu video, pausas aquí y explicas por qué el modelo no deduce que "lo de siempre" es la ubicación.

Cargando modelo...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Modelo cargado exitosamente.

=== DEMOSTRACIÓN DE MEJORA (Frase 38) ===
Input: Oye, agéndame una junta con la Dani pasado mañana a las 4 en la sala B-11

-> Ejecutando BASELINE (Direct Prompting)...
Salida Baseline:
{
  "accion": "agendar_reunion",
  "participantes": [
    "Dani"
  ],
  "fecha": "pasado mañana",
  "hora": "16:00",
  "lugar": "sala B-11",
  "mensaje": "Oye, agéndame una junta con la Dani pasado mañana a las 4 en la sala B-11"
}

-> Ejecutando SOLUCIÓN (System Prompt + Fecha Inyectada)...
Salida Solución:
{
  "titulo": "Junta con la Dani",
  "fecha": "2026-09-03",
  "hora_inicio": "16:00",
  "duracion_minutos": null,
  "participantes": [
    "Dani"
  ],
  "ubicacion": "Sala B-11",
  "recordatorio_minutos": null,
  "zona_horaria": "America/Santiago"
}

=== LECTURA DE LÍMITES (Frase 40 - Failure Case) ===
Input: q onda, quedamos el viernes a las 7 en lo de siempre?

Salida Solución en caso límite:
{
  "titulo": "Quedarnos el viernes a las 7 en lo de siempre",
  "fecha": "2

In [2]:
import json
import re
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

# ==========================================
# 1. CONFIGURACIÓN Y CARGA DE MODELO
# ==========================================
model_id = "Qwen/Qwen3-4B-Instruct-2507"
ARCHIVO_JSON = "frases.json" # Asegúrate de haberlo subido a Colab

print(f"Cargando {model_id} en GPU...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("¡Modelo cargado exitosamente!\n")

# Variables de entorno (Ancla temporal requerida por tu tarea)
FECHA_HOY = "2026-09-01"
DIA_HOY = "martes"

# ==========================================
# 2. DEFINICIÓN DE ESTRATEGIAS (Baseline vs Intervención)
# ==========================================
def baseline_prompt(texto):
    """Falla diagnosticada: Prompting directo colapsa sin esquema."""
    prompt = f'Agendar esta reunión y devolver JSON: "{texto}"'
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
    return tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

def intervention_prompt(texto):
    """Solución: System prompt (reglas + ancla temporal) + Few-Shot (formato) + Constrained Parsing."""
    system_prompt = f"""Eres un asistente de agendamiento. Hoy es {DIA_HOY}, {FECHA_HOY}. Resuelve fechas relativas basándote en hoy.
Extrae la info a JSON con campos: titulo, fecha (YYYY-MM-DD), hora_inicio (HH:MM), duracion_minutos, participantes (lista de nombres explícitos), ubicacion, recordatorio_minutos, zona_horaria (America/Santiago por defecto).
Usa null o [] si falta información. No asumas ni inventes datos. Responde SOLO con el JSON válido."""

    few_shot_user = "Reunión mañana a las 10"
    few_shot_assistant = """{
  "titulo": "Reunión",
  "fecha": "2026-09-02",
  "hora_inicio": "10:00",
  "duracion_minutos": null,
  "participantes": [],
  "ubicacion": null,
  "recordatorio_minutos": null,
  "zona_horaria": "America/Santiago"
}"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": few_shot_user},
        {"role": "assistant", "content": few_shot_assistant},
        {"role": "user", "content": texto}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
    respuesta = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

    # Post-procesamiento
    json_match = re.search(r'\{.*\}', respuesta, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except:
            return {"error": "JSON malformado"}
    return {"error": "No emitió JSON", "raw": respuesta}

# ==========================================
# 3. MODO VIDEO (3 MINUTOS)
# ==========================================
print("==================================================")
print("  INICIO GRABACIÓN PARA VIDEO - DEMONSTRATED EXECUTION")
print("==================================================\n")

frase_demo = "Oye, agéndame una junta con la Dani pasado mañana a las 4 en la sala B-11"
print(f"[INPUT]: {frase_demo}\n")

print("1. Ejecutando BASELINE (Direct Prompting)...")
print(baseline_prompt(frase_demo))
print("\n--------------------------------------------------\n")

print("2. Ejecutando SOLUCIÓN (System Prompt + Fecha Inyectada)...")
json_solucion = intervention_prompt(frase_demo)
print(json.dumps(json_solucion, indent=2, ensure_ascii=False))
print("\n--------------------------------------------------\n")

frase_limite = "q onda, quedamos el viernes a las 7 en lo de siempre?"
print(f"3. LECTURA DE LÍMITES (Failure Case): {frase_limite}")
json_limite = intervention_prompt(frase_limite)
print(json.dumps(json_limite, indent=2, ensure_ascii=False))
print("\n-> NOTA: El modelo omite 'lo de siempre' como ubicación, mostrando el límite de su razonamiento semántico.\n")
print("==================================================")
print("  FIN DE MODO VIDEO")
print("==================================================\n")

# ==========================================
# 4. MODO REPOSITORIO (EVALUACIÓN MASIVA PARA EL PDF)
# ==========================================
print("Iniciando evaluación masiva de las 45 frases del benchmark para el PDF técnico...")
try:
    with open(ARCHIVO_JSON, "r", encoding="utf-8") as f:
        datos = json.load(f)

    resultados_finales = []

    # Evaluar las primeras 5 frases como prueba (para evaluar las 45 completas, cambia el [:5] por datos['frases'])
    for item in datos['frases']:
        print(f"Procesando ID {item['id']}...")
        salida_json = intervention_prompt(item['texto'])
        resultados_finales.append({
            "id": item['id'],
            "texto": item['texto'],
            "ground_truth": item['ground_truth'],
            "resultado_modelo": salida_json
        })

    with open("resultados_deliverable2.json", "w", encoding="utf-8") as f:
        json.dump(resultados_finales, f, indent=2, ensure_ascii=False)

    print("\n¡Evaluación completa! Los resultados se han guardado en 'resultados_deliverable2.json'")
    print("Descarga este archivo para calcular tus métricas finales para el PDF.")

except FileNotFoundError:
    print(f"\n[ERROR] No se encontró el archivo '{ARCHIVO_JSON}'. Sube el archivo a Colab para correr el pipeline completo.")

Cargando Qwen/Qwen3-4B-Instruct-2507 en GPU...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

¡Modelo cargado exitosamente!

  INICIO GRABACIÓN PARA VIDEO - DEMONSTRATED EXECUTION

[INPUT]: Oye, agéndame una junta con la Dani pasado mañana a las 4 en la sala B-11

1. Ejecutando BASELINE (Direct Prompting)...
{
  "accion": "agendar_reunion",
  "participantes": [
    "Dani"
  ],
  "fecha": "pasado mañana",
  "hora": "16:00",
  "lugar": "sala B-11",
  "mensaje": "Oye, agéndame una junta con la Dani pasado mañana a las 4 en la sala B-11"
}

--------------------------------------------------

2. Ejecutando SOLUCIÓN (System Prompt + Fecha Inyectada)...
{
  "titulo": "Junta con la Dani",
  "fecha": "2026-09-03",
  "hora_inicio": "16:00",
  "duracion_minutos": null,
  "participantes": [
    "Dani"
  ],
  "ubicacion": "sala B-11",
  "recordatorio_minutos": null,
  "zona_horaria": "America/Santiago"
}

--------------------------------------------------

3. LECTURA DE LÍMITES (Failure Case): q onda, quedamos el viernes a las 7 en lo de siempre?
{
  "titulo": "Quedamos el viernes a las 7",